# Linear Probing 평가 — STL10 / CIFAR10 (Jupyter 서버)

**순서**
1. Cell 1: 작업 디렉토리(레포 루트) + GPU 확인
2. Cell 2: Feature 추출 (backbone → .npy)
3. Cell 3: evaluate.py 실행 (linear probing — 고정 recipe, 수정 금지)

MoCo v2 / MoCo v3 각각 Cell 2~3 실행하면 됨. (Drive 마운트/코드 복사 단계는 서버에서 불필요)

In [ ]:
# Cell 1 — 작업 디렉토리 = 레포 루트 + GPU 확인
import os, torch
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
print('Working dir:', os.getcwd())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2 — Feature 추출
# METHOD 변경: 'mocov2' 또는 'mocov3'
import glob, os, subprocess

METHOD = 'mocov2'   # ← 여기만 바꾸면 됨

OUTPUT_DIRS = {
    'mocov2': 'outputs/mocov2_r50_seed42',
    'mocov3': 'outputs/mocov3_vits_seed42',
}
CONFIG_FILES = {
    'mocov2': 'configs/mocov2_r50.yaml',
    'mocov3': 'configs/mocov3_vits.yaml',
}


# 마지막 backbone 체크포인트 자동 탐색
backbone_ckpts = sorted(
    glob.glob(f'{OUTPUT_DIRS[METHOD]}/backbone_ep*.pth'),
    key=lambda p: int(p.split('backbone_ep')[1].replace('.pth', ''))
)
if not backbone_ckpts:
    raise FileNotFoundError(f'{OUTPUT_DIRS[METHOD]}/ 에 backbone_ep*.pth 파일이 없습니다.')

backbone_path = backbone_ckpts[-1]
feature_dir   = f'outputs/{METHOD}_features'

print(f'Method   : {METHOD}')
print(f'Backbone : {backbone_path}')
print(f'Feature  : {feature_dir}/')
print()

proc = subprocess.Popen(
    [
        'python3', '-u', 'scripts/extract_features.py',
        '--backbone',    backbone_path,
        '--config',      CONFIG_FILES[METHOD],
        '--output-dir',  feature_dir,
        '--data-dir',    './data',
        '--batch-size',  '512',
        '--num-workers', '2',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nFeature 추출 완료 (exit={proc.returncode})')

In [ ]:
# Cell 3 — Linear Probing 평가
import os, subprocess

METHOD = 'mocov2'   # ← Cell 2 와 동일하게 맞출 것

feature_dir = f'outputs/{METHOD}_features'

proc = subprocess.Popen(
    [
        'python3', '-u', 'evaluate.py',
        '--stl10-train-features',    f'{feature_dir}/stl10_train_features.npy',
        '--stl10-train-labels',      f'{feature_dir}/stl10_train_labels.npy',
        '--stl10-test-features',     f'{feature_dir}/stl10_test_features.npy',
        '--stl10-test-labels',       f'{feature_dir}/stl10_test_labels.npy',
        '--cifar10-train-features',  f'{feature_dir}/cifar10_train_features.npy',
        '--cifar10-train-labels',    f'{feature_dir}/cifar10_train_labels.npy',
        '--cifar10-test-features',   f'{feature_dir}/cifar10_test_features.npy',
        '--cifar10-test-labels',     f'{feature_dir}/cifar10_test_labels.npy',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n평가 완료 (exit={proc.returncode})')